# CopyLane 판정 인코더 파인튜닝 v11 — KcBERT · Colab 원클릭 (소성민, 2026-10-06)

골든은 v10 과 같은 판이다 — **재동결 10-05 (8,383행 · 학습 5,894 · 평가 2,489 · sha `529c9705…`)**. 파일을 새로 받을 필요가 없다.
10-06 에 main 으로 들어온 결정 둘(**D-296 개정 2 · D-321**)을 반영해 다시 학습한다. **학습 설정은 v10 과 같다.**

## ▶ 실행 방법

1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** (이미 GPU 면 건너뛴다)
2. 메뉴 **런타임 → 모두 실행**
3. 처음에 Google Drive 연결 창이 뜨면 **허용**한다 — 공유 저장소(`CopyLane_store`)에서 `golden.jsonl` · `banned_terms.jsonl` 을 **sha 로 찾아** 가져온다.
   Drive 에서 못 찾으면 업로드 창이 뜬다 → 로컬의 두 파일을 올린다.

| 파일 | 로컬 위치 (`data-sync` 로 받은 것) |
|---|---|
| `golden.jsonl` | `C:\SKN32-FINAL-3TEAM\data\derived\golden\golden.jsonl` (sha `529c9705…`) |
| `banned_terms.jsonl` | `C:\SKN32-FINAL-3TEAM\data\derived\banned_terms.jsonl` (sha `455f8d7e…`) |

끝나면 **결과 보고서(.docx) · 결과 묶음(.zip) · 모델(.zip)** 이 자동으로 내려받아진다. 브라우저가 「여러 파일 다운로드」를 물으면 허용한다.
Drive 가 연결돼 있으면 같은 파일이 `내 드라이브/CopyLane_runs/<실행 이름>/` 에도 남는다.

- 설정은 바꾸지 않는다. 바꿀 칸이 없다 — 백본만 다른 노트북이 따로 있다(kcbert · kcelectra).
- 세션이 살아 있는 동안 다시 **모두 실행**하면 끝난 시드는 건너뛴다.
- 🔴 **test 는 이 노트북이 고른 모델 하나로 한 번만 채점한다(D-175).** 이 결과로 설정 · threshold 를 다시 고르지 않는다. 백본 둘의 우열은 **dev 수치**로 말한다.
- ⚠️ 이 골든 판의 test 는 v10 에서 이미 한 번 채점했다. v11 채점은 같은 판의 **재채점**이고, 보고서에 그렇게 적힌다.
- 산출물에는 재배포 불가 문구가 섞여 있다. 팀 내부 밖으로 올리지 않는다(D-303).

## v10 → v11 에서 바뀐 것

| # | 바뀐 것 | 근거 |
|:-:|---|---|
| 1 | **조건 M 행을 학습에서 뺀다** — 양성도 음성도 아니다. v10 은 M 을 음성으로 가르쳤다. D · L 은 그대로 음성이다 | D-296 개정 2 · 팀장 전달 10-05 §3 |
| 2 | **확정 클래스 6종 → 8종** — `부당_비교광고` · `비방광고` 를 출력에 넣는다. threshold 는 같은 규칙(dev F1 최대)으로 고른다 | D-321 결정 2 |
| 3 | **`기능성화장품_오인` 은 편입 대기** — 칸은 저장하되(모두 아홉 칸) threshold 를 1.01 로 두어 내지 않는다. 0.5 기준 수치는 참고값으로만 적는다 | D-321 결정 3 |
| 4 | **모델을 고르는 기준(epoch · 시드)은 종전 6종의 dev macro PR-AUC 그대로다.** 결과 표는 6종 기준 · 8종 기준을 둘 다 낸다 | D-321 결정 4 |
| 5 | 보고서에 비방 ↔ 부당 비교 혼동 수 · 유형 없는 위반 행(평가 131)의 탐지율을 따로 적는다 | 전달 10-06 보탬 |
| 6 | 모델과 함께 저장하는 정보(`label_scheme.json`)에 **조건별 학습 행 수**와 M 제외 여부를 적는다 | 전달 10-05 §4-3 |

그대로 둔 것 — dev 분리(원천별 15% · 시드 42 · v10 과 같은 묶음) · 유형 미정 C·A·B 행은 학습에서 뺌(근거가 빈 5행 포함) · 사례집 낱말 행은 학습에 둠 · 추가 학습 파일 없음.

학습 설정은 그대로다 — Focal γ1.5 · pos_weight √(음/양) 상한 5 · 배치 32 · warmup 10% · 시드 42 · 43 · 44 · threshold 는 고른 시드의 dev F1 최대.
백본별 값: **kcbert** lr 2e-5 · 최대 6 epoch · patience 2 / **kcelectra** lr 3e-5 · 최대 10 epoch · patience 3 (v10 과 같다).

## 0. 설정 — 고정값 (바꾸지 않는다)

In [ ]:
BACKBONE = "kcbert"

BACKBONES = {
    "kcbert":    dict(model="beomi/kcbert-base",     lr=2e-5, max_epochs=6,  patience=2),   # D-70
    "kcelectra": dict(model="beomi/KcELECTRA-base",  lr=3e-5, max_epochs=10, patience=3),   # D-84 비교군 · v5(09-29) lr
}
SEEDS = [42, 43, 44]
BATCH, WARMUP, WEIGHT_DECAY, MAX_LEN, GRAD_CLIP = 32, 0.1, 0.01, 128, 1.0
FOCAL_GAMMA, POS_WEIGHT_CAP = 1.5, 5
DEV_FRAC, DEV_SEED = 0.15, 42
NOTEBOOK_VERSION = "v11"

# D-176 — 입력 지문. main afd4940 의 data/derived_manifest.jsonl 값이다 (재동결 10-05 · D-318 · v10 과 같은 판)
EXPECT = {
    "golden.jsonl":       ("529c970556d77bd2c231526e20531294bf469f794e74a4796835a36f456eb7fe", 8383),
    "banned_terms.jsonl": ("455f8d7ea5c227ffab268a33cef9bc36750b0144f073a2ed33e1f7298dc5b892", 506),
}
REPO_COMMIT = "afd4940 (main · 2026-10-06)"
STORE_SHORTCUT_ID = "1_3HrAhrrLgy6jxAPprGGLjDQvpIsQ2h9"   # CopyLane_store (Drive 공유 저장소)

# D-321 (2026-10-06) — 확정 클래스 8종 = 종전 6종 + 부당_비교광고 · 비방광고 (scripts/collect.py VIOLATION_TYPES 순서)
SEL = ["질병_예방치료_표방", "건강기능식품_오인", "의약품_오인", "거짓_과장", "소비자_기만", "후기_체험기_기만"]   # 종전 6종 — 모델을 고르는 기준(D-321 결정 4)
CONFIRMED = SEL + ["부당_비교광고", "비방광고"]              # 확정 8종 — 출력 · 채점
AUX = ["기능성화장품_오인"]                                   # 편입 대기 — 칸은 저장하고 출력은 꺼 둔다(D-321 결정 3)
WAIT_THRESHOLD, WAIT_REF_THRESHOLD = 1.01, 0.5               # 문턱 > 1 이면 내지 않는다 · 0.5 는 참고값
# D-296 개정 2 (2026-10-05) — C · A · B 는 양성 · D · L 은 음성 · M 은 학습에서 뺀다
POSITIVE_CONDS, NONPOSITIVE_CONDS, EXCLUDED_CONDS = ("C", "A", "B"), ("M", "D", "L"), ("M",)

# v10 (같은 골든 판 · M 을 음성으로 학습 · 6종) 수치 — 보고서의 참고 줄에만 쓴다
V10_REF = {"kcbert":    "양성 탐지 재현율 75.5% · D 플래그율 51.0% · test macro PR-AUC 0.540 · dev macro PR-AUC 0.756 ± 0.007",
           "kcelectra": "양성 탐지 재현율 84.0% · D 플래그율 60.1% · test macro PR-AUC 0.468 · dev macro PR-AUC 0.692 ± 0.005"}

## 1. 환경 · 입력 파일 (sha 대조 · D-176)

In [ ]:
import os, sys, json, hashlib, shutil, gc, math, random, time, glob, re, subprocess, datetime, zipfile
from collections import Counter, defaultdict
import numpy as np

try:
    from google.colab import files as colab_files, drive as colab_drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False
SMOKE = os.environ.get("COPYLANE_SMOKE") == "1"          # 개발용 자체 점검 — Colab 에서는 켜지지 않는다
WORK = "/content" if IN_COLAB else os.getcwd()
os.chdir(WORK)

try:
    import docx  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-docx"], check=False)

import torch, transformers, sklearn
CFG = BACKBONES[BACKBONE]
MODEL_NAME = os.environ.get("COPYLANE_MODEL_OVERRIDE", CFG["model"])
LR, MAX_EPOCHS, PATIENCE = CFG["lr"], CFG["max_epochs"], CFG["patience"]
if SMOKE:
    MAX_EPOCHS = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU"
print(f"torch {torch.__version__} · transformers {transformers.__version__} · 장치 {GPU_NAME}")
if IN_COLAB and DEVICE != "cuda":
    raise SystemExit("🔴 GPU 가 없다 — 메뉴 「런타임 → 런타임 유형 변경」에서 T4 GPU 를 고른 뒤 「모두 실행」을 다시 누른다")

KST = datetime.timezone(datetime.timedelta(hours=9))
T_START = time.time()
RUN_STAMP = datetime.datetime.now(KST).strftime("%Y%m%d-%H%M")
RUN_ID = f"{NOTEBOOK_VERSION}_{BACKBONE}_{RUN_STAMP}"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

# ── Drive 연결 (입력 찾기 + 결과 보관). 실패해도 계속 간다 — 업로드 창으로 받는다
DRIVE_OK = False
if IN_COLAB:
    try:
        colab_drive.mount("/content/drive", force_remount=False)
        DRIVE_OK = os.path.isdir("/content/drive/MyDrive")
    except Exception as e:
        print("Drive 연결 안 됨 — 업로드로 받는다:", str(e)[:120])

def drive_candidates(sha):
    tail = f"copylane-derived/objects/{sha[:2]}/{sha}"
    roots = [f"/content/drive/.shortcut-targets-by-id/{STORE_SHORTCUT_ID}/CopyLane_store",
             "/content/drive/MyDrive/CopyLane_store"]
    roots += glob.glob("/content/drive/MyDrive/*/CopyLane_store") + glob.glob("/content/drive/Shareddrives/*/CopyLane_store")
    return [os.path.join(r, tail) for r in roots]

def find_input(name):
    """이름이 아니라 내용(sha)으로 찾는다 — `golden (1).jsonl` 같은 이름도 잡는다."""
    want = EXPECT[name][0]
    for p in sorted(glob.glob(os.path.join(WORK, "*.jsonl"))):
        if sha256(p) == want:
            return p, "작업 폴더"
    if DRIVE_OK:
        for p in drive_candidates(want):
            if os.path.isfile(p) and sha256(p) == want:
                dst = os.path.join(WORK, name)
                shutil.copy(p, dst)
                return dst, "Drive 공유 저장소(CopyLane_store)"
    return None, None

INPUTS, SOURCES = {}, {}
for _name in EXPECT:
    INPUTS[_name], SOURCES[_name] = find_input(_name)
_missing = [n for n, p in INPUTS.items() if p is None]
if _missing and IN_COLAB:
    print(f"⬆ 업로드 창에서 {' · '.join(_missing)} 을(를) 고른다 (한 번에 여러 파일 선택 가능)")
    colab_files.upload()
    for _name in _missing:
        INPUTS[_name], SOURCES[_name] = find_input(_name)
        if INPUTS[_name]:
            SOURCES[_name] = "직접 업로드"
_missing = [n for n, p in INPUTS.items() if p is None]
if _missing:
    _seen = {os.path.basename(p): sha256(p)[:12] for p in glob.glob(os.path.join(WORK, "*.jsonl"))}
    _want = {n: EXPECT[n][0][:12] for n in _missing}
    raise SystemExit(f"🔴 원장과 sha 가 같은 파일을 찾지 못했다 — {_missing} (D-176)\n   기대 sha: {_want}\n"
                     f"   작업 폴더에 있는 파일: {_seen}\n"
                     "   로컬에서 `uv run python launcher.py data-sync` 로 최신 판을 받은 뒤 그 파일을 올린다.")
for _name, _p in INPUTS.items():
    print(f"✅ {_name:20s} sha {EXPECT[_name][0][:12]}…  ← {SOURCES[_name]}")
GOLDEN_SHA = EXPECT["golden.jsonl"][0]

OUT_DIR = os.path.join(WORK, "CopyLane_runs", RUN_ID)
CKPT_DIR = os.path.join(WORK, f"ckpt_{NOTEBOOK_VERSION}_{BACKBONE}_{GOLDEN_SHA[:8]}" + ("_smoke" if SMOKE else ""))
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(CKPT_DIR, exist_ok=True)
DRIVE_RUNS = "/content/drive/MyDrive/CopyLane_runs" if DRIVE_OK else None
print("실행 이름:", RUN_ID)

## 2. 데이터 로드 · 채점 버킷

| 조건 | 학습 목표 | 채점 버킷 | 재는 것 |
|---|---|---|---|
| C · A · B (확정 8종 라벨) | 그 유형 양성 | `scored` | 유형별 P · R · F1 · PR-AUC · **양성 탐지 재현율** |
| C · A · B (유형 미정) | 학습에서 뺌 | `pending` | **양성 탐지 재현율** (따로도 적는다) |
| C · A · B (편입 대기 라벨 — 기능성화장품 오인) | 대기 칸 양성 | `aux` | 참고 (threshold 0.5) |
| L | 음성 | `neg` | **음성 오판정률** |
| D | 음성 | `D` | D 플래그율 (별도 지표) |
| M | **학습에서 뺌** (D-296 개정 2) | `M` | 참고 (채점 안 함) |

버킷은 두 벌이다 — `bucket`(확정 8종 기준 · D-321)과 `bucket6`(종전 6종 기준 · v10 과 견주는 표).

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

golden = load_jsonl(INPUTS["golden.jsonl"])
assert len(golden) == EXPECT["golden.jsonl"][1], f"행 수가 다르다: {len(golden)}"

PROV_KO = {"ftc_decisions_body": "공정위 의결서", "ftc_press": "공정위 보도자료",
           "mfds_special_use_guide": "특수용도식품 해설서", "mfds_cosmetic_ad_qa": "화장품 광고 질의응답집",
           "mfds_hf_ingredient_board": "승인 기능성 문구 게시판", "mfds_casebook": "식약처 사례집(낱말)"}
prov_ko = lambda p: PROV_KO.get(p, p)

def norm_text(s):
    return " ".join(s.split())

def group_key(r):
    rid = r["id"]
    if rid.startswith("inj:"):              # 'inj:T1:hf:0:0' → 원본 승인 문구 'hf:0:0'
        return rid.split(":", 2)[2]
    return rid.split("#")[0]

def normalize(r):
    basis = r.get("근거") or []
    cond = r.get("조건")
    labels = list(r.get("labels") or [])
    return {"id": r["id"], "group": group_key(r), "text": r["text"], "labels": labels,
            # 🔴 조건 M · D · L 은 라벨 · 근거가 있어도 양성이 아니다. M 은 학습에서 아예 뺀다(3절 · D-296 개정 2) — 여기 target 은 dev 채점용
            "target": [] if cond in NONPOSITIVE_CONDS else labels,
            "split": r["split"], "provenance": r.get("provenance"), "origin": r.get("origin"),
            "zone": r.get("구역"), "cond": cond, "unit": r.get("unit"),
            "hos": sorted({c.split("|")[0] for c in basis}), "laws": sorted({c.split(":")[0] for c in basis})}

all_rows = [normalize(r) for r in golden]
unknown = {l for r in all_rows for l in r["labels"]} - set(CONFIRMED) - set(AUX)
assert not unknown, f"🔴 예상 밖 라벨 — 라벨 목록이 바뀌었다: {unknown}"
assert {r["cond"] for r in all_rows} <= set(POSITIVE_CONDS) | set(NONPOSITIVE_CONDS) | {None}, "🔴 예상 밖 조건 값"
assert {r["split"] for r in all_rows} == {"train", "test_sentence"}, {r["split"] for r in all_rows}

def bucket(r, types):
    """types 밖의 라벨이 하나라도 있으면 aux — 6종 기준에서는 v10 과 같은 버킷이 된다."""
    if r["cond"] == "M": return "M"
    if r["cond"] == "D": return "D"
    if r["cond"] == "L":
        assert not r["labels"], f"🔴 조건 L(적법)인데 라벨이 있다 — {r['id']}"
        return "neg"
    if any(l not in types for l in r["labels"]): return "aux"
    if r["labels"]: return "scored"
    return "pending" if r["cond"] in POSITIVE_CONDS else "neg"

for r in all_rows:
    r["bucket"] = bucket(r, CONFIRMED)      # 확정 8종 기준 (D-321)
    r["bucket6"] = bucket(r, SEL)           # 종전 6종 기준 (v10 과 견주는 표)
train_all = [r for r in all_rows if r["split"] == "train"]
test_rows = [r for r in all_rows if r["split"] != "train"]
BUCKET_NAMES = ["scored", "pending", "aux", "neg", "D", "M"]
by_bucket = lambda rows, key="bucket": {b: [r for r in rows if r[key] == b] for b in BUCKET_NAMES}
TEST_B = by_bucket(test_rows)
print(f"golden {len(all_rows)}행 · train {len(train_all)} · test {len(test_rows)}")
print("train 조건:", dict(Counter(str(r["cond"]) for r in train_all)))
print("test 버킷(8종 기준):", {b: len(v) for b, v in TEST_B.items()})
print("test 버킷(6종 기준):", {b: len(v) for b, v in by_bucket(test_rows, "bucket6").items()})
TRAIN_PROVS = {r["provenance"] for r in train_all}

## 3. dev 분리 · 학습셋

1. dev — 학습 원천마다 묶음(문서 · 원본 문구)의 15% 를 뗀다. 채점은 그 묶음의 **문장 · 주입 아닌 행**으로 한다(낱말과 문장을 섞지 않는다 · 주입 행은 템플릿이라 점수를 부풀린다). **v10 과 같은 묶음이다.**
2. dev 묶음의 행은 전부 학습에서 빠진다. dev 와 글자가 같은 학습 행도 뺀다. test 와 같은 문구는 빼지 않는다(D-174).
3. 🆕 **조건 M 행을 학습에서 뺀다**(D-296 개정 2). dev 의 M 행은 남겨 참고 지표(M 플래그율)로만 본다.
4. 유형 미정 C·A·B 행을 뺀다. 글자와 학습 목표가 모두 같은 학습 행은 하나로 합친다.

In [ ]:
def dev_eligible(r):
    return r["unit"] == "문장" and r["origin"] != "injected"

groups = defaultdict(list)
for r in train_all:
    groups[r["group"]].append(r)

# ── 원천별 15% (묶음 단위 · 시드 고정)
prov_groups = defaultdict(list)
for g in sorted(groups):
    elig = [r for r in groups[g] if dev_eligible(r)]
    if elig:
        prov_groups[Counter(r["provenance"] for r in elig).most_common(1)[0][0]].append(g)
rng = random.Random(DEV_SEED)
dev_groups = set()
for prov in sorted(prov_groups):
    gs = prov_groups[prov][:]
    rng.shuffle(gs)
    dev_groups.update(gs[:max(1, math.ceil(DEV_FRAC * len(gs)))])

# ── 유형마다 dev 양성이 너무 적으면(10 미만) 채운다 — threshold 를 고를 수 있게
def conf_pos(rows, l):
    return sum(1 for r in rows if dev_eligible(r) and l in r["target"])
rest = [g for g in sorted(groups) if g not in dev_groups]
rng.shuffle(rest)
for l in SEL:                               # v10 과 같은 dev 묶음이 나오도록 종전 6종으로만 채운다
    total = conf_pos(train_all, l)
    want = min(10, math.ceil(DEV_FRAC * total))
    have = sum(conf_pos(groups[g], l) for g in dev_groups)
    for g in rest:
        if have >= want:
            break
        k = conf_pos(groups[g], l)
        if k and g not in dev_groups:
            dev_groups.add(g); have += k

dev_rows = [r for g in sorted(dev_groups) for r in groups[g] if dev_eligible(r)]
dev_texts = {norm_text(r["text"]) for r in dev_rows}
DEV_B = by_bucket(dev_rows)

# ── 학습 풀
pool = [r for r in train_all if r["group"] not in dev_groups]
n_pool0 = len(pool)
pool = [r for r in pool if norm_text(r["text"]) not in dev_texts]
n_dev_dup = n_pool0 - len(pool)
n_M = sum(1 for r in pool if r["cond"] in EXCLUDED_CONDS)
pool = [r for r in pool if r["cond"] not in EXCLUDED_CONDS]   # 🆕 조건 M — 양성도 음성도 아니다 (D-296 개정 2)
n_pending = sum(1 for r in pool if r["bucket"] == "pending")
pool = [r for r in pool if r["bucket"] != "pending"]          # 유형 미정 위반 — 음성으로 가르치지 않는다
seen, train_rows = set(), []
for r in pool:
    k = (norm_text(r["text"]), tuple(sorted(r["target"])))
    if k in seen:
        continue
    seen.add(k); train_rows.append(r)
n_merged = len(pool) - len(train_rows)
if SMOKE:
    random.Random(0).shuffle(train_rows); train_rows = train_rows[:500]
    dev_rows = dev_rows[::3]; DEV_B = by_bucket(dev_rows)
    test_rows = test_rows[::6]; TEST_B = by_bucket(test_rows)

LABEL_LIST = CONFIRMED + AUX
LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
N_SEL, N_CONF, N_ALL = len(SEL), len(CONFIRMED), len(LABEL_LIST)
assert LABEL_LIST[:N_SEL] == SEL and not any(r["cond"] == "M" for r in train_rows)

def multihot(labels):
    v = np.zeros(N_ALL, dtype=np.float32)
    for l in labels:
        v[LABEL2ID[l]] = 1.0
    return v

Y_train = np.stack([multihot(r["target"]) for r in train_rows])
Y_dev = np.stack([multihot(r["target"]) for r in dev_rows]).astype(int)

DATA_STATS = {
    "train_all": len(train_all), "dev_groups": len(dev_groups), "dev_rows": len(dev_rows),
    "dev_buckets": {b: len(v) for b, v in DEV_B.items()},
    "dev_buckets6": {b: len(v) for b, v in by_bucket(dev_rows, "bucket6").items()},
    "dev_injected_rows": sum(1 for r in dev_rows if r["origin"] == "injected"),
    "removed_dev_dup": n_dev_dup, "removed_M": n_M, "removed_pending": n_pending, "merged_dup": n_merged,
    "train_rows": len(train_rows),
    "train_cond": dict(Counter(str(r["cond"]) for r in train_rows)),
    "train_pos_rows": int((Y_train.sum(1) > 0).sum()), "train_neg_rows": int((Y_train.sum(1) == 0).sum()),
    "train_nonpositive_with_labels": sum(1 for r in train_rows if r["cond"] in NONPOSITIVE_CONDS and r["labels"]),
    "train_reason_rows": sum(1 for r in train_rows if r["zone"] == "이유"),
    "train_label_pos": {l: int(Y_train[:, i].sum()) for i, l in ID2LABEL.items()},
    "train_label_injected": {l: sum(1 for r in train_rows if l in r["target"] and r["origin"] == "injected") for l in LABEL_LIST},
    "train_label_word": {l: sum(1 for r in train_rows if l in r["target"] and r["unit"] == "낱말") for l in LABEL_LIST},
    "dev_label_pos": {l: int(Y_dev[:, i].sum()) for i, l in ID2LABEL.items()},
    "train_by_prov": dict(Counter(r["provenance"] for r in train_rows)),
    "dev_by_prov": dict(Counter(r["provenance"] for r in dev_rows)),
    "test_rows": len(test_rows), "test_buckets": {b: len(v) for b, v in TEST_B.items()},
    "test_buckets6": {b: len(v) for b, v in by_bucket(test_rows, "bucket6").items()},
}
print(f"dev: 묶음 {len(dev_groups)}개 · 채점 {len(dev_rows)}행 · 버킷 {DATA_STATS['dev_buckets']}")
print(f"train: {len(train_all)} → dev 묶음 뺌 {n_pool0} → dev 와 같은 문구 −{n_dev_dup} → 조건 M −{n_M} → 유형 미정 −{n_pending} → 중복 병합 −{n_merged} → {len(train_rows)}행")
print("  조건별 학습 행:", DATA_STATS["train_cond"])
print(f"  양성 행 {DATA_STATS['train_pos_rows']} · 음성 행 {DATA_STATS['train_neg_rows']} "
      f"(음성은 D · L 과 승인 문구 — 라벨이 적혀 있던 D·L {DATA_STATS['train_nonpositive_with_labels']}행 포함)")
print(f"{'유형':20s} {'train 양성':>9s} {'dev 양성':>8s}")
for i, l in ID2LABEL.items():
    print(f"  {l:20s} {DATA_STATS['train_label_pos'][l]:8d} {DATA_STATS['dev_label_pos'][l]:8d}" + ("" if i < N_SEL else ("  (🆕 편입)" if i < N_CONF else "  (편입 대기 — 출력 끔)")))

## 4. 토크나이저 · 손실 · 지표

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, precision_recall_fscore_support

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
PAD_ID = tokenizer.pad_token_id

def encode(rows):
    return tokenizer([r["text"] for r in rows], truncation=True, max_length=MAX_LEN)["input_ids"]

def batchify(ids_list, idx):
    m = max(len(ids_list[i]) for i in idx)
    ids = torch.full((len(idx), m), PAD_ID, dtype=torch.long)
    att = torch.zeros((len(idx), m), dtype=torch.long)
    for k, i in enumerate(idx):
        ids[k, :len(ids_list[i])] = torch.tensor(ids_list[i]); att[k, :len(ids_list[i])] = 1
    return ids.to(DEVICE), att.to(DEVICE)

def amp():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda"))

@torch.no_grad()
def predict_probs(model, ids_list, bs=128):
    model.eval()
    out = []
    order = np.argsort([len(x) for x in ids_list])
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        ids, att = batchify(ids_list, idx)
        with amp():
            lg = model(input_ids=ids, attention_mask=att).logits
        out.append((idx, torch.sigmoid(lg.float()).cpu().numpy()))
    p = np.zeros((len(ids_list), model.config.num_labels), dtype=np.float32)
    for idx, v in out:
        p[idx] = v
    return p

X_train, X_dev = encode(train_rows), encode(dev_rows)

pos = Y_train.sum(0); neg = len(Y_train) - pos
POS_WEIGHT = np.clip(np.sqrt(neg / np.clip(pos, 1e-6, None)), 1, POS_WEIGHT_CAP)
print("pos_weight:", {l: round(float(POS_WEIGHT[i]), 2) for i, l in ID2LABEL.items()})
_pw = torch.tensor(POS_WEIGHT, dtype=torch.float32, device=DEVICE)

def focal_loss(logits, labels):
    logits = logits.float()
    bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels, reduction="none")
    p = torch.sigmoid(logits)
    p_t = p * labels + (1 - p) * (1 - labels)
    return ((_pw * labels + (1 - labels)) * (1 - p_t) ** FOCAL_GAMMA * bce).mean()

def safe(fn, y, p):
    return float(fn(y, p)) if 0 < y.sum() < len(y) else None

def conf_metrics(y, p, types=SEL):
    """기본은 종전 6종 — 모델을 고르는 기준이다(D-321 결정 4). macro 는 양성이 있는 유형의 평균."""
    out, prs, rocs = {}, [], []
    for i, l in enumerate(types):
        pr, roc = safe(average_precision_score, y[:, i], p[:, i]), safe(roc_auc_score, y[:, i], p[:, i])
        out[f"prauc_{l}"], out[f"rocauc_{l}"] = pr, roc
        if pr is not None:
            prs.append(pr); rocs.append(roc)
    out["macro_prauc"] = float(np.mean(prs)) if prs else float("nan")
    out["macro_rocauc"] = float(np.mean(rocs)) if rocs else float("nan")
    return out

TH_GRID = np.arange(0.1, 0.9001, 0.025)
def pick_thresholds(y, p):
    """dev 유형별 F1 최대 · 격자 0.1~0.9(0.025) · dev 양성 10건 미만이면 0.5 (v7.4 · v9 · v10 과 같은 규칙) · 확정 8종에 적용."""
    th = {}
    for i, l in enumerate(CONFIRMED):
        if y[:, i].sum() < 10:
            th[l] = 0.5; continue
        fs = [f1_score(y[:, i], p[:, i] >= t, zero_division=0) for t in TH_GRID]
        th[l] = float(round(TH_GRID[int(np.argmax(fs))], 3))
    return th

def wilson(k, n, z=1.96):
    """Wilson 95% 구간 (D-40)."""
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if DEVICE == "cuda":
        torch.cuda.manual_seed_all(seed)

## 5. 학습 — 시드마다 한 번 (dev macro PR-AUC · **종전 6종 기준**으로 조기 종료)

In [ ]:
def train_seed(seed):
    sdir = os.path.join(CKPT_DIR, f"seed{seed}")
    done = os.path.join(sdir, "done.json")
    if os.path.exists(done):
        res = json.load(open(done, encoding="utf-8"))
        res["dev_p"] = np.load(os.path.join(sdir, "dev_p.npy"))
        print(f"seed {seed}: 이미 끝남 — 건너뜀 (dev macro PR-AUC {res['dev']['macro_prauc']:.3f})")
        return res
    shutil.rmtree(sdir, ignore_errors=True); os.makedirs(sdir)
    t0 = time.time()
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=N_ALL, problem_type="multi_label_classification",
        id2label=ID2LABEL, label2id=LABEL2ID).to(DEVICE)
    no_decay = ("bias", "LayerNorm.weight", "LayerNorm.bias")
    params = [{"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)], "weight_decay": WEIGHT_DECAY},
              {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(params, lr=LR)
    spe = math.ceil(len(train_rows) / BATCH)
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * spe * MAX_EPOCHS), spe * MAX_EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))
    Yt = torch.tensor(Y_train, device=DEVICE)
    rs = np.random.RandomState(seed)
    best, best_epoch, bad, curve = -1.0, 0, 0, []
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        perm = rs.permutation(len(train_rows)); tot = 0.0
        for s in range(0, len(perm), BATCH):
            idx = perm[s:s + BATCH]
            ids, att = batchify(X_train, idx)
            with amp():
                logits = model(input_ids=ids, attention_mask=att).logits
            loss = focal_loss(logits, Yt[torch.as_tensor(idx, device=DEVICE)])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt); scaler.update(); sched.step()
            tot += float(loss)
        dm = conf_metrics(Y_dev, predict_probs(model, X_dev))
        curve.append([epoch, round(dm["macro_prauc"], 4), round(tot / spe, 5)])
        print(f"  seed {seed} · epoch {epoch}: 손실 {tot / spe:.4f} · dev macro PR-AUC {dm['macro_prauc']:.4f}"
              + ("  ★" if dm["macro_prauc"] > best else ""))
        if dm["macro_prauc"] > best:
            best, best_epoch, bad = dm["macro_prauc"], epoch, 0
            model.save_pretrained(os.path.join(sdir, "model"), safe_serialization=True)
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    del model, opt, scaler; gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    bm = AutoModelForSequenceClassification.from_pretrained(os.path.join(sdir, "model")).to(DEVICE)
    dev_p = predict_probs(bm, X_dev)
    del bm; gc.collect()
    res = {"seed": seed, "dev": conf_metrics(Y_dev, dev_p), "dev8": conf_metrics(Y_dev, dev_p, CONFIRMED), "best_epoch": best_epoch, "epochs_run": len(curve),
           "curve": curve, "thresholds": pick_thresholds(Y_dev, dev_p), "dir": os.path.join(sdir, "model"),
           "minutes": round((time.time() - t0) / 60, 1)}
    np.save(os.path.join(sdir, "dev_p.npy"), dev_p)
    json.dump(res, open(done, "w", encoding="utf-8"), ensure_ascii=False)
    res["dev_p"] = dev_p
    print(f"seed {seed}: dev macro PR-AUC 6종 {res['dev']['macro_prauc']:.3f} · 8종 {res['dev8']['macro_prauc']:.3f} (best epoch {best_epoch}/{len(curve)}) · {res['minutes']}분")
    return res

SEED_RES = {}
for _seed in SEEDS:
    print(f"\n==================== {BACKBONE} · seed {_seed} ====================")
    SEED_RES[_seed] = train_seed(_seed)

DEV_PRAUCS = [SEED_RES[s]["dev"]["macro_prauc"] for s in SEEDS]
DEV_PRAUCS8 = [SEED_RES[s]["dev8"]["macro_prauc"] for s in SEEDS]
# 🔒 선택 규칙 — test 를 보기 전에 고정: dev macro PR-AUC(종전 6종 · D-321 결정 4)가 가장 높은 시드 · threshold 는 그 시드의 dev F1 최대
BEST_SEED = max(SEEDS, key=lambda s: SEED_RES[s]["dev"]["macro_prauc"])
TH = SEED_RES[BEST_SEED]["thresholds"]
print(f"\n[{BACKBONE}] dev macro PR-AUC 6종 {np.mean(DEV_PRAUCS):.3f} ± {np.std(DEV_PRAUCS):.3f} · 8종 {np.mean(DEV_PRAUCS8):.3f} ± {np.std(DEV_PRAUCS8):.3f} · 선택 시드 {BEST_SEED}")
print("threshold:", TH)

## 6. 채점 — dev (선택 모델) · test (**1회** · D-175)

같은 함수로 dev 와 test 를 잰다. 모델은 한 번만 돌리고, 같은 예측을 **확정 8종 기준**(D-321)과 **종전 6종 기준**(v10 과 견주는 표) 두 벌로 센다. 후보 · 백본 비교는 dev 수치로 한다.

In [ ]:
best_model = AutoModelForSequenceClassification.from_pretrained(SEED_RES[BEST_SEED]["dir"]).to(DEVICE)

def rate(k, n):
    lo, hi = wilson(k, n)
    return {"k": int(k), "n": int(n), "rate": (k / n if n else None), "lo": lo, "hi": hi}

def evaluate(rows, P_all, name, types, bkey):
    """types — 채점할 유형(LABEL_LIST 의 앞 6칸 또는 8칸) · bkey — 그 기준의 버킷 칸."""
    nt = len(types)
    assert LABEL_LIST[:nt] == list(types)
    th = np.array([TH[l] for l in types])
    P = P_all[:, :nt] if len(rows) else np.zeros((0, nt))
    flag = P >= th
    anyflag = flag.any(1)
    B = defaultdict(list)
    for i, r in enumerate(rows):
        B[r[bkey]].append(i)
    idx = {b: np.array(B[b], dtype=int) for b in BUCKET_NAMES}
    res = {"n": len(rows), "types": list(types), "buckets": {b: len(idx[b]) for b in BUCKET_NAMES}}

    # ── 팀장 채점 (C·A·B 양성 · L 음성 · D 별도 · M 채점 안 함)
    pos_i = np.concatenate([idx["scored"], idx["pending"]])
    res["team"] = {"pos_recall": rate(anyflag[pos_i].sum(), len(pos_i)), "L_fpr": rate(anyflag[idx["neg"]].sum(), len(idx["neg"])),
                   "D_flag": rate(anyflag[idx["D"]].sum(), len(idx["D"])), "M_flag": rate(anyflag[idx["M"]].sum(), len(idx["M"])),
                   "aux_flag": rate(anyflag[idx["aux"]].sum(), len(idx["aux"])),
                   "scored_recall": rate(anyflag[idx["scored"]].sum(), len(idx["scored"])),
                   "pending_recall": rate(anyflag[idx["pending"]].sum(), len(idx["pending"]))}
    # ── 플래그를 어느 유형이 울렸나 (D · M · L · aux — 선택에는 쓰지 않는다)
    res["flag_by_type"] = {b: {l: int(flag[idx[b], i].sum()) for i, l in enumerate(types)} for b in ["D", "M", "neg", "aux"]}

    # ── 유형별 (scored)
    S = idx["scored"]
    y = np.stack([multihot(rows[i]["labels"]) for i in S])[:, :nt].astype(int) if len(S) else np.zeros((0, nt), int)
    p, f = P[S], flag[S].astype(int)
    m = conf_metrics(y, p, types) if len(S) else {"macro_prauc": float("nan"), "macro_rocauc": float("nan")}
    res["macro_prauc"], res["macro_rocauc"] = m["macro_prauc"], m["macro_rocauc"]
    res["label"] = {}
    for i, l in enumerate(types):
        n = int(y[:, i].sum()); tp = int((f[:, i] & y[:, i]).sum()); pp = int(f[:, i].sum())
        P_, R_ = (tp / pp if pp else 0.0), (tp / n if n else 0.0)
        res["label"][l] = {"n": n, "th": TH[l], "tp": tp, "pred": pp, "P": P_, "R": R_,
                           "F1": (2 * P_ * R_ / (P_ + R_) if P_ + R_ else 0.0),
                           "R_ci": wilson(tp, n), "P_ci": wilson(tp, pp),
                           "prauc": m.get(f"prauc_{l}"), "rocauc": m.get(f"rocauc_{l}")}
    if len(S):
        _, _, mf, _ = precision_recall_fscore_support(y, f, average="micro", zero_division=0)
        res["micro_f1"] = float(mf)
        meas = [l for l in types if res["label"][l]["n"] >= 30]
        res["macro_f1_measurable"] = float(np.mean([res["label"][l]["F1"] for l in meas])) if meas else None
        res["measurable"] = meas
        hit = (f & y).any(1)
    else:
        res["micro_f1"], res["macro_f1_measurable"], res["measurable"], hit = None, None, [], np.zeros(0, bool)

    # ── 원천별 · 조건별 · 호별
    srows = [rows[i] for i in S]
    res["by_prov"] = {}
    for prov in sorted({r["provenance"] for r in rows}):
        si = [k for k, r in enumerate(srows) if r["provenance"] == prov]
        pi = [i for i in pos_i if rows[i]["provenance"] == prov]
        res["by_prov"][prov] = {"hit": rate(hit[si].sum() if si else 0, len(si)), "pos_recall": rate(anyflag[pi].sum() if pi else 0, len(pi)),
                                "in_train": prov in TRAIN_PROVS}
    res["by_cond"] = {}
    for c in ["C", "A", "B"]:
        si = [k for k, r in enumerate(srows) if r["cond"] == c]
        pi = [i for i in pos_i if rows[i]["cond"] == c]
        res["by_cond"][c] = {"hit": rate(hit[si].sum() if si else 0, len(si)), "pos_recall": rate(anyflag[pi].sum() if pi else 0, len(pi))}
    res["by_ho"] = {}
    for ho in sorted({h for r in srows for h in r["hos"]}):
        si = [k for k, r in enumerate(srows) if ho in r["hos"]]
        res["by_ho"][ho] = rate(hit[si].sum(), len(si))

    print(f"\n════ {name} · {len(rows)}행 · 선택 모델({BACKBONE} · 시드 {BEST_SEED}) ════")
    t = res["team"]
    for title, key, note in [("양성 탐지 재현율 (C·A·B)", "pos_recall", "높을수록 좋다"), ("  └ 유형 없는 위반 행만", "pending_recall", "따로 적는다"),
                             ("음성 오판정률 (L)", "L_fpr", "낮을수록 좋다"),
                             ("D 플래그율 (별도 지표)", "D_flag", "낮을수록 좋다"), ("M 플래그율 (참고)", "M_flag", "채점 안 함")]:
        v = t[key]
        if v["n"]:
            print(f"  {title:24s} {v['k']:5d}/{v['n']:<5d} {v['rate']*100:5.1f}% [{v['lo']*100:.1f}~{v['hi']*100:.1f}%]  {note}" + ("  · 측정 불가(n<30)" if v["n"] < 30 else ""))
    if len(S):
        print(f"  macro PR-AUC {res['macro_prauc']:.3f} · macro ROC-AUC {res['macro_rocauc']:.3f} · micro F1 {res['micro_f1']:.3f}")
        print(f"  {'유형':20s} {'n':>5s} {'th':>6s} {'P':>6s} {'R':>6s} {'F1':>6s} {'PR-AUC':>7s}")
        for l in types:
            v = res["label"][l]
            print(f"  {l:20s} {v['n']:5d} {v['th']:6.3f} {v['P']:6.3f} {v['R']:6.3f} {v['F1']:6.3f} " + (f"{v['prauc']:7.3f}" if v["prauc"] is not None else "    N/A") + ("  측정 불가(n<30)" if v["n"] < 30 else ""))
    return res, {"rows": rows, "S": S, "y": y, "flag": f}

def cross_confusion(rows, P_all):
    """비방 ↔ 부당 비교 — 한쪽만 정답인 채점 행에서 무엇이 울렸나 (D-321 ⬜ · 식품 6호와 7호는 경계가 겹친다)."""
    a, b = "부당_비교광고", "비방광고"
    fa, fb = P_all[:, LABEL2ID[a]] >= TH[a], P_all[:, LABEL2ID[b]] >= TH[b]
    out = {}
    for gold, other, fg, fo in [(a, b, fa, fb), (b, a, fb, fa)]:
        ix = np.array([i for i, r in enumerate(rows) if r["bucket"] == "scored" and gold in r["labels"] and other not in r["labels"]], dtype=int)
        out[gold] = {"n": len(ix), "own_only": int((fg[ix] & ~fo[ix]).sum()), "both": int((fg[ix] & fo[ix]).sum()),
                     "other_only": int((~fg[ix] & fo[ix]).sum()), "neither": int((~fg[ix] & ~fo[ix]).sum()), "other": other}
    return out

def wait_reference(rows, P_all):
    """편입 대기 유형 — threshold 0.5 의 참고값 (M 행 제외 · 게이트 · 발표 지표에 넣지 않는다 · D-321 결정 3)."""
    out = {}
    for l in AUX:
        ix = np.array([i for i, r in enumerate(rows) if r["cond"] != "M"], dtype=int)
        y = np.array([int(l in rows[i]["target"]) for i in ix]); p = P_all[ix, LABEL2ID[l]] if len(ix) else np.zeros(0)
        f = (p >= WAIT_REF_THRESHOLD).astype(int)
        n, tp, pp = int(y.sum()), int((f & y).sum()), int(f.sum())
        out[l] = {"n": n, "tp": tp, "pred": pp, "P": (tp / pp if pp else 0.0), "R": (tp / n if n else 0.0),
                  "prauc": safe(average_precision_score, y, p) if len(ix) else None, "th": WAIT_REF_THRESHOLD}
    return out

P_DEV = predict_probs(best_model, X_dev)
DEV_RES, _ = evaluate(dev_rows, P_DEV, "dev · 확정 8종 기준", CONFIRMED, "bucket")
DEV_RES6, _ = evaluate(dev_rows, P_DEV, "dev · 종전 6종 기준", SEL, "bucket6")
DEV_CROSS, DEV_WAIT = cross_confusion(dev_rows, P_DEV), wait_reference(dev_rows, P_DEV)

# ── test 채점 횟수 기록 (D-175) — 같은 골든 판을 몇 번 채점했는지 남긴다
TEST_LOG_PATH = os.path.join(DRIVE_RUNS, "test_채점_기록.json") if DRIVE_RUNS else None
prior_tests = []
if TEST_LOG_PATH and os.path.exists(TEST_LOG_PATH):
    try:
        prior_tests = [e for e in json.load(open(TEST_LOG_PATH, encoding="utf-8")) if e.get("golden_sha256") == GOLDEN_SHA]
    except Exception:
        prior_tests = []
PRIOR_SAME_BACKBONE = [e for e in prior_tests if e.get("backbone") == BACKBONE]
print(f"\n⚠️ 이 골든 판의 test 는 v10 에서 이미 채점했다 — 이번은 재채점이고 보고서에 그렇게 적는다 (D-175 · Drive 기록 {len(PRIOR_SAME_BACKBONE)}건)")
print("🔴 test 최종 측정 — 이 결과로 설정 · 데이터 · threshold 를 다시 고르지 않는다 (D-175)")
P_TEST = predict_probs(best_model, encode(test_rows))          # 모델은 test 를 한 번만 본다 — 아래 두 표는 같은 예측을 다르게 센 것
TEST_RES, TEST_AUX = evaluate(test_rows, P_TEST, "test · 확정 8종 기준", CONFIRMED, "bucket")
TEST_RES6, _ = evaluate(test_rows, P_TEST, "test · 종전 6종 기준 (v10 과 견주는 표)", SEL, "bucket6")
TEST_CROSS, TEST_WAIT = cross_confusion(test_rows, P_TEST), wait_reference(test_rows, P_TEST)
for _nm, _cr in [("dev", DEV_CROSS), ("test", TEST_CROSS)]:
    for _g, _v in _cr.items():
        print(f"  [{_nm}] 정답 {_g}만 {_v['n']}행 → 제 유형만 {_v['own_only']} · 둘 다 {_v['both']} · {_v['other']}만 {_v['other_only']} · 안 울림 {_v['neither']}")
for _l, _v in TEST_WAIT.items():
    print(f"  [test · 참고] {_l} (threshold {_v['th']}) n {_v['n']} · P {_v['P']:.3f} · R {_v['R']:.3f} — 편입 대기 · 저장 모델에서는 나오지 않는다")


## 7. 기준선 B(사전 단독판정) + G1 게이트

G1: 모델 재현율(dev threshold) ≥ 사전 단독판정 재현율 + 0.15. 정답 30건 미만은 참고용(D-40). 확정 8종 기준 채점 행으로 잰다.

In [ ]:
banned = load_jsonl(INPUTS["banned_terms.jsonl"])
assert len(banned) == EXPECT["banned_terms.jsonl"][1], len(banned)
solo = [b for b in banned if b.get("단독판정") is True]
term_labels = [(s, set(b.get("유형", [])) & set(CONFIRMED)) for b in solo for s in b.get("원문", []) if s]

def rule_predict(text):
    out = set()
    for s, ls in term_labels:
        if s in text:
            out |= ls
    return out

GATE_MARGIN, MIN_N = 0.15, 30
GATE = []
_rows, _S, _y, _f = TEST_AUX["rows"], TEST_AUX["S"], TEST_AUX["y"], TEST_AUX["flag"]
_rule = [rule_predict(_rows[i]["text"]) for i in _S]
print(f"사전 {len(banned)}종 · 단독판정 {len(solo)}종\n{'유형':20s} {'기준선B':>8s} {'모델R':>7s} {'통과선':>7s} {'n':>5s}  판정")
for i, l in enumerate(CONFIRMED):
    ks = [k for k in range(len(_S)) if _y[k, i]]
    if not ks:
        continue
    b = sum(l in _rule[k] for k in ks) / len(ks)
    m = float(_f[ks, i].mean())
    verdict = f"참고용(n<{MIN_N})" if len(ks) < MIN_N else ("통과" if m >= b + GATE_MARGIN else "미달")
    GATE.append({"label": l, "baseline_recall": b, "model_recall": m, "pass_line": b + GATE_MARGIN, "n": len(ks), "verdict": verdict})
    print(f"{l:20s} {b:8.3f} {m:7.3f} {b + GATE_MARGIN:7.3f} {len(ks):5d}  {verdict}")

## 8. 모델 저장 (앱 로더 형식)

고른 시드의 가중치를 **아홉 칸 그대로** 저장한다 — 확정 8종 + 편입 대기 1칸(`기능성화장품_오인` · threshold 1.01 이라 앱에서 나오지 않는다 · D-321 결정 3).
`config.json` 의 `id2label` 과 `label_scheme.json` 의 `label_list` 순서가 같다(`app/encoder.py` 가 대조한다).

In [ ]:
MODEL_DIRNAME = f"copylane-encoder-{BACKBONE}-final"
SAVE_DIR = os.path.join(OUT_DIR, MODEL_DIRNAME)
shutil.rmtree(SAVE_DIR, ignore_errors=True)
assert best_model.config.num_labels == N_ALL and [best_model.config.id2label[i] for i in range(N_ALL)] == LABEL_LIST
best_model.cpu().float().save_pretrained(SAVE_DIR, safe_serialization=True)
best_model.to(DEVICE)
tokenizer.save_pretrained(SAVE_DIR)
assert os.path.exists(os.path.join(SAVE_DIR, "tokenizer.json")), "tokenizer.json 이 없다 (앱 로더 필수 파일)"
TH_SAVE = {**{l: TH[l] for l in CONFIRMED}, **{l: WAIT_THRESHOLD for l in AUX}}

def jsonable(o):
    if isinstance(o, dict): return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): return o.item()
    if isinstance(o, float) and o != o: return None
    if isinstance(o, np.ndarray): return o.tolist()
    return o

LABEL_SCHEME = {
    "label_list": LABEL_LIST, "label_thresholds": TH_SAVE, "confirmed_labels": CONFIRMED, "selection_labels": SEL,
    "pending_labels": AUX, "pending_reference_threshold": WAIT_REF_THRESHOLD, "backbone": MODEL_NAME, "notebook": NOTEBOOK_VERSION, "run_id": RUN_ID,
    "seed": BEST_SEED, "best_epoch": SEED_RES[BEST_SEED]["best_epoch"], "golden_sha256": GOLDEN_SHA,
    "banned_terms_sha256": EXPECT["banned_terms.jsonl"][0],
    "threshold_note": "선택 시드의 dev F1 최대 · 격자 0.1~0.9(0.025) · dev 양성 10건 미만이면 0.5. 적용은 확률 >= threshold.",
    "thresholds_by_seed": {str(s): SEED_RES[s]["thresholds"] for s in SEEDS},
    "threshold_wait_note": "편입 대기 유형(pending_labels)은 threshold 1.01 — 확률이 넘을 수 없어 나오지 않는다(D-321 결정 3).",
    "train_note": "새 판 · M 제외 — 조건 M 행은 학습에서 뺐다(D-296 개정 2). D · L 은 음성. 유형 미정 C·A·B 행은 학습에서 뺐다. 추가 학습 파일 없음.",
    "train_m_excluded": True, "train_removed_M": DATA_STATS["removed_M"], "train_rows": DATA_STATS["train_rows"],
    "train_cond_rows": DATA_STATS["train_cond"], "train_label_pos": DATA_STATS["train_label_pos"],
    "selection_note": "epoch · 시드는 종전 6종(selection_labels)의 dev macro PR-AUC 로 골랐다(D-321 결정 4).",
    "scope_note": "확정 8종(D-321). 판정 대상 아님(D-275) · 보류(M)는 이 인코더가 내지 않는다.",
    "dev_team": DEV_RES["team"], "test_team": TEST_RES["team"], "dev_team_6": DEV_RES6["team"], "test_team_6": TEST_RES6["team"], "final_test": True,
}
json.dump(jsonable(LABEL_SCHEME), open(os.path.join(SAVE_DIR, "label_scheme.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=2)
with open(os.path.join(SAVE_DIR, "SHA256SUMS.txt"), "w", encoding="utf-8") as f:
    for fn in sorted(os.listdir(SAVE_DIR)):
        if fn != "SHA256SUMS.txt":
            f.write(f"{sha256(os.path.join(SAVE_DIR, fn))}  {fn}\n")
print("저장:", sorted(os.listdir(SAVE_DIR)), f"· {N_ALL}로짓(확정 {N_CONF} + 대기 {len(AUX)})")
print("threshold:", TH_SAVE)

ELAPSED_MIN = round((time.time() - T_START) / 60, 1)
RESULTS = jsonable({
    "run_id": RUN_ID, "notebook": NOTEBOOK_VERSION, "backbone": BACKBONE, "model_name": MODEL_NAME, "repo_commit": REPO_COMMIT,
    "inputs": {n: {"sha256": EXPECT[n][0], "rows": EXPECT[n][1], "source": SOURCES[n]} for n in EXPECT},
    "env": {"torch": torch.__version__, "transformers": transformers.__version__, "device": GPU_NAME, "elapsed_min": ELAPSED_MIN},
    "config": {"lr": LR, "max_epochs": MAX_EPOCHS, "patience": PATIENCE, "batch": BATCH, "warmup": WARMUP, "weight_decay": WEIGHT_DECAY,
               "max_len": MAX_LEN, "focal_gamma": FOCAL_GAMMA, "pos_weight_cap": POS_WEIGHT_CAP, "seeds": SEEDS,
               "dev_frac": DEV_FRAC, "dev_seed": DEV_SEED, "pos_weight": {l: float(POS_WEIGHT[i]) for i, l in ID2LABEL.items()}},
    "data": DATA_STATS,
    "seeds": {str(s): {k: v for k, v in SEED_RES[s].items() if k not in ("dev_p", "dir")} for s in SEEDS},
    "dev_macro_prauc_mean": float(np.mean(DEV_PRAUCS)), "dev_macro_prauc_std": float(np.std(DEV_PRAUCS)),
    "dev8_macro_prauc_mean": float(np.mean(DEV_PRAUCS8)), "dev8_macro_prauc_std": float(np.std(DEV_PRAUCS8)),
    "best_seed": BEST_SEED, "thresholds": TH_SAVE, "dev": DEV_RES, "test": TEST_RES, "dev_6": DEV_RES6, "test_6": TEST_RES6, "gate": GATE,
    "cross_confusion": {"dev": DEV_CROSS, "test": TEST_CROSS}, "pending_label_reference": {"dev": DEV_WAIT, "test": TEST_WAIT},
    "prior_test_scorings_same_golden": prior_tests, "smoke": SMOKE,
})
json.dump(RESULTS, open(os.path.join(OUT_DIR, "results.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=2)
if TEST_LOG_PATH and not SMOKE:
    try:
        _log = json.load(open(TEST_LOG_PATH, encoding="utf-8")) if os.path.exists(TEST_LOG_PATH) else []
        _log.append({"golden_sha256": GOLDEN_SHA, "backbone": BACKBONE, "run_id": RUN_ID, "seed": BEST_SEED, "notebook": NOTEBOOK_VERSION,
                     "pos_recall": TEST_RES["team"]["pos_recall"]["rate"], "macro_prauc": TEST_RES["macro_prauc"]})
        json.dump(jsonable(_log), open(TEST_LOG_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    except Exception as e:
        print("test 채점 기록 저장 안 됨:", e)

## 9. 결과 보고서 (.docx) · 그림

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager

# ── 한글 글꼴 (없으면 깔고, 그래도 없으면 유형 이름을 짧은 영문으로)
def find_kfont():
    for pat in ["/usr/share/fonts/truetype/nanum/NanumGothic.ttf", "/usr/share/fonts/**/NanumGothic*.ttf",
                "/usr/share/fonts/**/NotoSansCJK*Regular*.ttc", "/usr/share/fonts/**/NotoSansCJK*.ttc", "C:/Windows/Fonts/malgun.ttf"]:
        hit = sorted(glob.glob(pat, recursive=True))
        if hit:
            return hit[0]
    return None
KFONT = find_kfont()
if KFONT is None and IN_COLAB:
    subprocess.run("apt-get -qq -y install fonts-nanum > /dev/null 2>&1", shell=True)
    KFONT = find_kfont()
if KFONT:
    font_manager.fontManager.addfont(KFONT)
    plt.rcParams["font.family"] = font_manager.FontProperties(fname=KFONT).get_name()
plt.rcParams["axes.unicode_minus"] = False
EN = {"질병_예방치료_표방": "disease", "건강기능식품_오인": "health-food", "의약품_오인": "drug", "거짓_과장": "false/exagg.",
      "소비자_기만": "deception", "후기_체험기_기만": "review", "부당_비교광고": "unfair-comp.", "비방광고": "disparage"}
SHORT = {"질병_예방치료_표방": "질병", "건강기능식품_오인": "건기식", "의약품_오인": "의약품", "거짓_과장": "거짓", "소비자_기만": "기만",
         "후기_체험기_기만": "후기", "부당_비교광고": "부당비교", "비방광고": "비방"}
lab = lambda l: l.replace("_", "\n") if KFONT else EN.get(l, l)
tr = lambda ko, en: ko if KFONT else en
C_BLUE, C_ORANGE, C_AQUA, INK, MUTED, GRID, SURF = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"

def style(ax):
    ax.set_facecolor(SURF)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.tick_params(colors=MUTED, length=0, labelsize=9)
    ax.set_axisbelow(True)

FIGS = {}
# 그림 1 — 유형별 PR-AUC · dev 와 test
fig, ax = plt.subplots(figsize=(8.2, 3.6), dpi=160); fig.patch.set_facecolor(SURF); style(ax)
x = np.arange(N_CONF); w = 0.36
dv = [DEV_RES["label"][l]["prauc"] or 0 for l in CONFIRMED]; tv = [TEST_RES["label"][l]["prauc"] or 0 for l in CONFIRMED]
ax.bar(x - w / 2 - 0.01, dv, w, color=C_BLUE, label=f"dev ({DEV_RES['buckets']['scored']})")
ax.bar(x + w / 2 + 0.01, tv, w, color=C_ORANGE, label=f"test ({TEST_RES['buckets']['scored']})")
ax.set_xticks(x); ax.set_xticklabels([lab(l) for l in CONFIRMED], color=INK, fontsize=8.5)
ax.set_ylim(0, 1.0); ax.yaxis.grid(True, color=GRID, linewidth=0.8); ax.set_ylabel("PR-AUC", color=MUTED, fontsize=9)
ax.set_title(tr("유형별 PR-AUC — dev 와 test", "PR-AUC by type — dev vs test"), loc="left", color=INK, fontsize=11)
ax.legend(frameon=False, fontsize=9, loc="lower right", bbox_to_anchor=(1.0, 1.0), ncol=2)
fig.tight_layout(); FIGS["prauc"] = os.path.join(OUT_DIR, "그림1_유형별_PR-AUC.png"); fig.savefig(FIGS["prauc"], facecolor=SURF); plt.close(fig)

# 그림 2 — 원천별 양성 탐지 재현율 (test)
_pv = [(p, v) for p, v in TEST_RES["by_prov"].items() if v["pos_recall"]["n"] > 0]
_pv.sort(key=lambda kv: kv[1]["pos_recall"]["rate"])
fig, ax = plt.subplots(figsize=(8.2, 0.42 * len(_pv) + 1.3), dpi=160); fig.patch.set_facecolor(SURF); style(ax)
yy = np.arange(len(_pv))
ax.barh(yy, [v["pos_recall"]["rate"] for _, v in _pv], 0.6, color=[C_BLUE if v["in_train"] else C_ORANGE for _, v in _pv])
for k, (_, v) in enumerate(_pv):
    ax.text(min(v["pos_recall"]["rate"] + 0.012, 1.0), k, f"{v['pos_recall']['rate']*100:.0f}%  (n={v['pos_recall']['n']})", va="center", fontsize=8.5, color=INK)
ax.set_yticks(yy); ax.set_yticklabels([(prov_ko(p) if KFONT else p) for p, _ in _pv], color=INK)
ax.set_xlim(0, 1.18); ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0]); ax.xaxis.grid(True, color=GRID, linewidth=0.8)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=C_BLUE, label=tr("학습에 있던 원천", "source seen in training")), Patch(color=C_ORANGE, label=tr("학습에 없던 원천", "unseen source"))],
          frameon=False, fontsize=9, loc="lower right", bbox_to_anchor=(1.0, 1.0), ncol=2)
ax.set_title(tr("test 원천별 양성 탐지 재현율 (C·A·B)", "test positive recall by source (C/A/B)"), loc="left", color=INK, fontsize=11)
fig.tight_layout(); FIGS["prov"] = os.path.join(OUT_DIR, "그림2_원천별_탐지율.png"); fig.savefig(FIGS["prov"], facecolor=SURF); plt.close(fig)

# 그림 3 — epoch 별 dev macro PR-AUC
fig, ax = plt.subplots(figsize=(8.2, 3.0), dpi=160); fig.patch.set_facecolor(SURF); style(ax)
for s, c in zip(SEEDS, [C_BLUE, C_ORANGE, C_AQUA]):
    cv = SEED_RES[s]["curve"]
    ax.plot([e[0] for e in cv], [e[1] for e in cv], color=c, linewidth=2, marker="o", markersize=5, label=tr(f"시드 {s}", f"seed {s}") + (" ★" if s == BEST_SEED else ""))
ax.set_xticks(range(1, max(len(SEED_RES[s]["curve"]) for s in SEEDS) + 1)); ax.yaxis.grid(True, color=GRID, linewidth=0.8)
ax.set_xlabel("epoch", color=MUTED, fontsize=9); ax.set_ylabel("dev macro PR-AUC", color=MUTED, fontsize=9)
ax.set_title(tr("epoch 별 dev macro PR-AUC", "dev macro PR-AUC by epoch"), loc="left", color=INK, fontsize=11)
ax.legend(frameon=False, fontsize=9, loc="lower right", bbox_to_anchor=(1.0, 1.0), ncol=3)
fig.tight_layout(); FIGS["curve"] = os.path.join(OUT_DIR, "그림3_epoch_곡선.png"); fig.savefig(FIGS["curve"], facecolor=SURF); plt.close(fig)

# ════════════════ 보고서 ════════════════
from docx import Document
from docx.shared import Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

FONT = "맑은 고딕"
doc = Document()
for sec in doc.sections:
    sec.left_margin = sec.right_margin = Cm(2.0); sec.top_margin = sec.bottom_margin = Cm(2.0)
def set_font(run, size=None, bold=None, color=None):
    run.font.name = FONT
    rpr = run._element.get_or_add_rPr()
    rf = rpr.find(qn("w:rFonts"))
    if rf is None:
        rf = OxmlElement("w:rFonts"); rpr.append(rf)
    rf.set(qn("w:eastAsia"), FONT)
    if size: run.font.size = Pt(size)
    if bold is not None: run.font.bold = bold
    if color: run.font.color.rgb = RGBColor.from_string(color)
def para(text="", size=10, bold=False, color=None, after=6, align=None):
    p = doc.add_paragraph(); p.paragraph_format.space_after = Pt(after); p.paragraph_format.space_before = Pt(0)
    if align: p.alignment = align
    if text: set_font(p.add_run(text), size, bold, color)
    return p
def h(text, level=1):
    return para(text, 14 if level == 1 else 11.5, True, "1F3864", after=6)
def bullet(text):
    p = doc.add_paragraph(style="List Bullet"); p.paragraph_format.space_after = Pt(2)
    set_font(p.add_run(text), 10); return p
def shade(cell, hexcolor):
    tcpr = cell._element.get_or_add_tcPr(); sh = OxmlElement("w:shd")
    sh.set(qn("w:val"), "clear"); sh.set(qn("w:color"), "auto"); sh.set(qn("w:fill"), hexcolor); tcpr.append(sh)
def table(header, rows, widths=None, size=9):
    t = doc.add_table(rows=1, cols=len(header)); t.style = "Table Grid"
    for j, tx in enumerate(header):
        c = t.rows[0].cells[j]; shade(c, "1F3864"); c.paragraphs[0].text = ""
        set_font(c.paragraphs[0].add_run(str(tx)), size, True, "FFFFFF")
    for row in rows:
        cells = t.add_row().cells
        for j, tx in enumerate(row):
            cells[j].paragraphs[0].text = ""
            set_font(cells[j].paragraphs[0].add_run(str(tx)), size)
    if widths:
        for row in t.rows:
            for j, wd in enumerate(widths):
                row.cells[j].width = Cm(wd)
    para("", after=4); return t
def image(path, width=16.5):
    doc.add_picture(path, width=Cm(width)); doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER

pct = lambda v: "—" if v is None or v != v else f"{v*100:.1f}%"
f3 = lambda v: "N/A" if v is None or v != v else f"{v:.3f}"
def rt(v, small="측정 불가 (n<30)"):
    if not v["n"]:
        return "—", "—", "행 없음"
    return f"{pct(v['rate'])} ({v['k']:,}/{v['n']:,})", f"{v['lo']*100:.1f}~{v['hi']*100:.1f}%", (small if v["n"] < 30 else "")

T, D = TEST_RES, DEV_RES
tt, dt = T["team"], D["team"]
passed = [g["label"] for g in GATE if g["verdict"] == "통과"]
gated = [g for g in GATE if not g["verdict"].startswith("참고")]
BB_KO = {"kcbert": "KcBERT (beomi/kcbert-base)", "kcelectra": "KcELECTRA (beomi/KcELECTRA-base)"}[BACKBONE]
today = datetime.datetime.now(KST).strftime("%Y-%m-%d")

para(f"판정 인코더 파인튜닝 결과 보고서 ({NOTEBOOK_VERSION} · {BACKBONE} · 재동결 10-05 · 확정 8종)", 17, True, "1F3864", after=2)
para(f"{today} · 소성민 · 실행 {RUN_ID}" + ("  ·  ⚠️ 자체 점검(SMOKE) 실행 — 수치는 의미 없음" if SMOKE else ""), 9.5, color="52514E", after=10)

h("요약")
para(f"재동결 10-05 판({EXPECT['golden.jsonl'][1]:,}행 · D-318 · v10 과 같은 판)으로 {BB_KO} 를 다시 학습했다. "
     f"조건 M {DATA_STATS['removed_M']:,}행을 학습에서 빼고(D-296 개정 2) 확정 클래스를 8종으로 넓혔다(D-321). 학습 설정은 v10 과 같다. "
     f"아래 수치는 확정 8종 기준이다 — test 양성 탐지 재현율은 {rt(tt['pos_recall'])[0]}, D 플래그율은 {rt(tt['D_flag'])[0]}, "
     f"macro PR-AUC 는 dev {f3(D['macro_prauc'])} · test {f3(T['macro_prauc'])} 다.")
para("팀장 채점 (test · 확정 8종 기준)", 10.5, True, after=3)
_rows = []
for title, key, judge in [("양성 탐지 재현율 (C·A·B)", "pos_recall", "높을수록 좋다"), ("음성 오판정률 (L)", "L_fpr", "낮을수록 좋다"),
                          ("D 플래그율 (별도 지표)", "D_flag", "낮을수록 좋다")]:
    a, b, c = rt(tt[key]); _rows.append([title, a, b, c or judge])
table(["지표", "값", "95% 구간", "판단"], _rows, [5.5, 4.5, 3.3, 3.7])
bullet(f"선택 모델: 시드 {BEST_SEED} · best epoch {SEED_RES[BEST_SEED]['best_epoch']} — test 를 보기 전에 정한 규칙(dev macro PR-AUC 최대)으로 골랐다.")
bullet(f"dev macro PR-AUC(종전 6종 · 선택 기준)는 시드 3개 평균 {np.mean(DEV_PRAUCS):.3f} ± {np.std(DEV_PRAUCS):.3f} 다. 백본 비교는 이 수로 한다. 8종 기준으로는 {np.mean(DEV_PRAUCS8):.3f} ± {np.std(DEV_PRAUCS8):.3f} 다.")
bullet(f"종전 6종 기준으로 세면 test 양성 탐지 재현율 {rt(TEST_RES6['team']['pos_recall'])[0]} · D 플래그율 {rt(TEST_RES6['team']['D_flag'])[0]} · macro PR-AUC dev {f3(DEV_RES6['macro_prauc'])} · test {f3(TEST_RES6['macro_prauc'])} 다 — v10 과 견줄 때는 이 줄을 쓴다.")
bullet(f"G1 게이트는 확정 8종 가운데 측정 가능한 {len(gated)}종 중 {len(passed)}종이 통과했다" + (f" ({' · '.join(passed)})." if passed else "."))
bullet("이 결과로 설정 · threshold 를 다시 고르지 않는다(D-175). 앱 모델 교체 여부는 팀장 판단이 필요하다.")
bullet("⚠️ 이 골든 판의 test 는 v10(10-06)에서 같은 백본으로 이미 한 번 채점했다 — 이번 수치는 재채점이다"
       + (f" (Drive 기록 {len(PRIOR_SAME_BACKBONE)}건)." if PRIOR_SAME_BACKBONE else "."))
para(f"참고 — v10(같은 골든 판 · M 을 음성으로 학습 · 6종 기준)은 {V10_REF[BACKBONE]} 였다. 학습 행이 달라졌으므로(M 제외) 6종 기준 표끼리만 견준다.", 9, color="52514E")

h("실행 조건")
_cfg = RESULTS["config"]
table(["항목", "값"], [
    ["실행", f"{'Colab' if IN_COLAB else '로컬'} · {GPU_NAME} · {RUN_STAMP} (KST) · 학습~저장 {ELAPSED_MIN}분 · torch {torch.__version__} · transformers {transformers.__version__}"],
    ["저장소 커밋", REPO_COMMIT],
    ["golden", f"재동결 10-05 · sha {GOLDEN_SHA[:12]} · {EXPECT['golden.jsonl'][1]:,}행 (train {len(train_all):,} · test {DATA_STATS['test_rows']:,}) · 받은 곳: {SOURCES['golden.jsonl']}"],
    ["사전", f"banned_terms sha {EXPECT['banned_terms.jsonl'][0][:12]} · {len(banned)}종 (단독판정 {len(solo)}종) — 기준선 B 용"],
    ["백본", BB_KO],
    ["학습 행", f"{DATA_STATS['train_rows']:,}행 — 양성 {DATA_STATS['train_pos_rows']:,} · 음성 {DATA_STATS['train_neg_rows']:,} (이유 구역 {DATA_STATS['train_reason_rows']:,}행 포함)"],
    ["학습 목표", "C · A · B 와 조건 칸 없는 행(사례집 낱말 · 주입)만 양성 · D · L 은 음성 · M 은 학습에서 뺌(D-296 개정 2)"],
    ["조건별 학습 행", " · ".join(f"{('칸 없음' if c == 'None' else c)} {n:,}" for c, n in sorted(DATA_STATS["train_cond"].items(), key=lambda kv: -kv[1]))],
    ["학습에서 뺀 행", f"dev 묶음 · dev 와 같은 문구 {DATA_STATS['removed_dev_dup']} · 조건 M {DATA_STATS['removed_M']:,} · 유형 미정 C·A·B {DATA_STATS['removed_pending']} · 중복 병합 {DATA_STATS['merged_dup']}"],
    ["dev", f"train 의 묶음 {DATA_STATS['dev_groups']}개(원천별 15% · v10 과 같은 묶음) · 주입(합성) 행 {DATA_STATS['dev_injected_rows']} · 채점 {DATA_STATS['dev_rows']:,}행 — 양성 {D['team']['pos_recall']['n']} · L {D['team']['L_fpr']['n']} · D {D['team']['D_flag']['n']} · M {D['team']['M_flag']['n']} · 편입 대기 {D['team']['aux_flag']['n']}"],
    ["학습 설정", f"Focal γ{FOCAL_GAMMA} · pos_weight √(음/양) 상한 {POS_WEIGHT_CAP} · lr {LR:g} · 최대 {MAX_EPOCHS} epoch · patience {PATIENCE} · 배치 {BATCH} · warmup {int(WARMUP*100)}% · 시드 {' · '.join(map(str, SEEDS))}"],
    ["threshold", "선택 시드의 dev F1 최대 (격자 0.1~0.9 · dev 양성 10건 미만이면 0.5) · 확정 8종에 같은 규칙"],
    ["선택 기준", "epoch · 시드는 종전 6종의 dev macro PR-AUC (D-321 결정 4 · v10 과 같다)"],
    ["출력", f"아홉 칸 저장 — 확정 8종 + 편입 대기 1칸(기능성화장품_오인 · threshold {WAIT_THRESHOLD} 로 꺼 둠 · D-321)"],
], [3.2, 13.8])
para("v10 대비 바꾼 것은 둘이다 — 조건 M 을 학습에서 뺐고(v10 은 음성), 출력을 확정 8종 + 대기 1칸으로 넓혔다. 손실 · 학습률 · epoch · 배치 · 시드 · threshold 규칙 · dev 묶음은 v10 과 같고, 추가 학습 파일과 시드 수프는 쓰지 않았다.", 9.5)

h("학습 데이터 구성")
table(["유형", "train 양성", "그중 주입(합성)", "그중 낱말", "dev 양성", "pos_weight"],
      [[l + ("" if i < N_SEL else (" (🆕 편입)" if i < N_CONF else " (편입 대기)")), f"{DATA_STATS['train_label_pos'][l]:,}", f"{DATA_STATS['train_label_injected'][l]:,}",
        f"{DATA_STATS['train_label_word'][l]:,}", f"{DATA_STATS['dev_label_pos'][l]:,}", f"{POS_WEIGHT[i]:.2f}"] for i, l in ID2LABEL.items()],
      [5.6, 2.3, 2.7, 2.0, 2.2, 2.2])
para("dev 에는 주입(합성) 행이 없다 — dev 수치는 전부 합성 행을 뺀 수치다.", 9, color="52514E")
_provs = sorted(set(DATA_STATS["train_by_prov"]) | set(DATA_STATS["dev_by_prov"]) | set(T["by_prov"]), key=lambda p: -DATA_STATS["train_by_prov"].get(p, 0))
_tp = Counter(r["provenance"] for r in test_rows)
table(["원천", "train 행", "dev 행", "test 행"],
      [[prov_ko(p), f"{DATA_STATS['train_by_prov'].get(p, 0):,}", f"{DATA_STATS['dev_by_prov'].get(p, 0):,}", f"{_tp.get(p, 0):,}"] for p in _provs],
      [8.0, 3.0, 3.0, 3.0])

h("dev 결과 (선택 기준)")
table(["시드", "dev macro PR-AUC (6종 · 선택 기준)", "dev macro PR-AUC (8종)", "best epoch / 돈 epoch", "소요(분)"],
      [[f"{s}" + (" (선택)" if s == BEST_SEED else ""), f3(SEED_RES[s]["dev"]["macro_prauc"]), f3(SEED_RES[s]["dev8"]["macro_prauc"]), f"{SEED_RES[s]['best_epoch']} / {SEED_RES[s]['epochs_run']}", SEED_RES[s].get("minutes", "—")] for s in SEEDS]
      + [["평균 ± 표준편차", f"{np.mean(DEV_PRAUCS):.3f} ± {np.std(DEV_PRAUCS):.3f}", f"{np.mean(DEV_PRAUCS8):.3f} ± {np.std(DEV_PRAUCS8):.3f}", "", ""]], [3.4, 4.2, 3.6, 3.4, 2.4])
image(FIGS["curve"])
para("선택 모델의 dev 팀장 채점 (확정 8종 기준)", 10.5, True, after=3)
_rows = []
for title, key in [("양성 탐지 재현율 (C·A·B)", "pos_recall"), ("음성 오판정률 (L)", "L_fpr"), ("D 플래그율", "D_flag"), ("M 플래그율 (참고)", "M_flag")]:
    a, b, c = rt(dt[key]); _rows.append([title, a, b, c])
table(["지표", "값", "95% 구간", "비고"], _rows, [5.5, 4.5, 3.3, 3.7])

h("최종 test 결과 (확정 8종 기준)")
para(f"고른 모델 하나를 test {T['n']:,}행에 한 번 채점했다. macro PR-AUC {f3(T['macro_prauc'])} · macro ROC-AUC {f3(T['macro_rocauc'])} · micro F1 {f3(T['micro_f1'])}"
     + (f" · 측정 가능 {len(T['measurable'])}종 macro F1 {f3(T['macro_f1_measurable'])}" if T.get("measurable") else "") + " 이다.")
para(f"유형별 (확정 8종 라벨이 있는 {T['buckets']['scored']:,}행 · 선택 모델 threshold)", 10.5, True, after=3)
_g = {g["label"]: g for g in GATE}
table(["유형", "n", "threshold", "정밀도", "재현율 (95% 구간)", "F1", "PR-AUC (dev → test)", "G1 게이트"],
      [[l, T["label"][l]["n"], f"{TH[l]:.3f}", f3(T["label"][l]["P"]),
        f"{f3(T['label'][l]['R'])} ({T['label'][l]['R_ci'][0]:.2f}~{T['label'][l]['R_ci'][1]:.2f})" if T["label"][l]["n"] else "—",
        f3(T["label"][l]["F1"]), f"{f3(D['label'][l]['prauc'])} → {f3(T['label'][l]['prauc'])}", _g[l]["verdict"] if l in _g else "—"] for l in CONFIRMED],
      [3.4, 1.2, 1.7, 1.5, 3.4, 1.4, 2.8, 1.9], size=8.5)
para("G1 게이트 = 모델 재현율 ≥ 사전 단독판정(기준선 B) 재현율 + 0.15. 정답 30건 미만은 참고용이다(D-40).", 9, color="52514E")
table(["유형", "기준선 B 재현율", "모델 재현율", "통과선", "n", "판정"],
      [[g["label"], f3(g["baseline_recall"]), f3(g["model_recall"]), f3(g["pass_line"]), g["n"], g["verdict"]] for g in GATE], [4.2, 2.8, 2.6, 2.2, 1.6, 3.2])
image(FIGS["prauc"])

para("원천별 (test)", 10.5, True, after=3)
table(["원천", "양성 탐지 재현율 (C·A·B)", "유형 적중률 (scored)", "학습에 있었나"],
      [[prov_ko(p), rt(v["pos_recall"])[0], rt(v["hit"])[0], "있음" if v["in_train"] else "없음"] for p, v in sorted(T["by_prov"].items(), key=lambda kv: -kv[1]["pos_recall"]["n"])],
      [5.5, 4.6, 4.3, 2.6])
para("유형 적중률 = 정답 유형 중 하나라도 맞힌 비율. 양성 탐지 재현율 = 확정 8종 중 하나라도 울린 비율.", 9, color="52514E")
image(FIGS["prov"])
para("조건별 (test)", 10.5, True, after=3)
table(["조건", "양성 탐지 재현율", "유형 적중률 (scored)"], [[c, rt(v["pos_recall"])[0], rt(v["hit"])[0]] for c, v in T["by_cond"].items()], [3.0, 7.0, 7.0])
para("호별 유형 적중률 (test · scored)", 10.5, True, after=3)
table(["호", "적중률", "95% 구간", "비고"], [[ho, *rt(v, "측정 불가 (호 n<30 · D-40)")] for ho, v in T["by_ho"].items()], [6.5, 4.0, 3.2, 3.3])

h("종전 6종 기준 (v10 과 견주는 표)")
para("같은 예측을 종전 6종으로만 센 것이다 — 부당 비교 · 비방 · 기능성화장품 라벨이 있는 행은 v10 처럼 채점 행에서 빠진다(D-321 결정 4).", 9.5)
_rows = []
for title, key in [("양성 탐지 재현율 (C·A·B)", "pos_recall"), ("음성 오판정률 (L)", "L_fpr"), ("D 플래그율", "D_flag"), ("M 플래그율 (참고)", "M_flag")]:
    a, b, _ = rt(TEST_RES6["team"][key]); a2, b2, _ = rt(DEV_RES6["team"][key]); _rows.append([title, a, b, a2, b2])
table(["지표", "test 값", "test 95% 구간", "dev 값", "dev 95% 구간"], _rows, [4.6, 3.6, 2.6, 3.6, 2.6], size=8.5)
para(f"macro PR-AUC — dev {f3(DEV_RES6['macro_prauc'])} · test {f3(TEST_RES6['macro_prauc'])} · test micro F1 {f3(TEST_RES6['micro_f1'])} (6종 라벨이 있는 test {TEST_RES6['buckets']['scored']:,}행 · dev {DEV_RES6['buckets']['scored']:,}행)", 9.5)
table(["유형", "n", "threshold", "정밀도", "재현율 (95% 구간)", "F1", "PR-AUC (dev → test)"],
      [[l, TEST_RES6["label"][l]["n"], f"{TH[l]:.3f}", f3(TEST_RES6["label"][l]["P"]),
        f"{f3(TEST_RES6['label'][l]['R'])} ({TEST_RES6['label'][l]['R_ci'][0]:.2f}~{TEST_RES6['label'][l]['R_ci'][1]:.2f})" if TEST_RES6["label"][l]["n"] else "—",
        f3(TEST_RES6["label"][l]["F1"]), f"{f3(DEV_RES6['label'][l]['prauc'])} → {f3(TEST_RES6['label'][l]['prauc'])}"] for l in SEL],
      [3.8, 1.3, 1.9, 1.7, 3.8, 1.5, 3.3], size=8.5)

h("새로 편입한 유형 · 편입 대기 · 유형 없는 위반")
para("비방 ↔ 부당 비교 혼동 — 둘 중 한쪽만 정답인 채점 행에서 무엇이 울렸나 (식품 6호 · 7호는 경계가 겹친다 · D-321)", 10.5, True, after=3)
table(["정답", "셋", "n", "제 유형만", "둘 다", "다른 쪽만 (혼동)", "안 울림"],
      [[g, nm, v["n"], v["own_only"], v["both"], v["other_only"], v["neither"]] for nm, cr in [("dev", DEV_CROSS), ("test", TEST_CROSS)] for g, v in cr.items()],
      [3.6, 1.4, 1.6, 2.4, 2.0, 3.4, 2.4])
para(f"편입 대기 — 기능성화장품_오인 (threshold {WAIT_REF_THRESHOLD} 의 참고값 · 게이트 · 발표 지표에 넣지 않는다)", 10.5, True, after=3)
table(["셋", "정답 n", "울린 수", "맞힌 수", "정밀도", "재현율", "PR-AUC"],
      [[nm, v["n"], v["pred"], v["tp"], f3(v["P"]), f3(v["R"]), f3(v["prauc"])] for nm, w in [("dev", DEV_WAIT), ("test", TEST_WAIT)] for v in w.values()],
      [2.0, 2.2, 2.2, 2.2, 2.6, 2.6, 3.0])
para("유형 없는 위반 행 (C·A·B 인데 유형 이름이 없는 행 — 화장품법 제13조① 4호 등 · 학습에서 뺐다)", 10.5, True, after=3)
_rows = []
for nm, R8 in [("test", T), ("dev", D)]:
    a, b, c = rt(R8["team"]["pending_recall"]); _rows.append([nm, a, b, c or "확정 8종 중 하나라도 울린 비율"])
table(["셋", "탐지율", "95% 구간", "비고"], _rows, [2.0, 5.0, 3.4, 6.4])
para("이 행들은 위의 「양성 탐지 재현율」 분모에 들어 있다(v10 과 같은 셈). 유형이 없어 유형별 표에는 없다.", 9, color="52514E")

h("플래그 내역 — 어느 유형이 울렸나")
para("채점 밖 버킷에서 확정 8종 중 무엇이 threshold 를 넘었는지 센 것이다(한 행이 여러 유형을 울릴 수 있다). 선택에는 쓰지 않는다.", 9.5)
_bn = [("D", "D (판정 대상 아님)", "D_flag"), ("M", "M (보류)", "M_flag"), ("neg", "L (적법)", "L_fpr"), ("aux", "편입 대기 라벨", "aux_flag")]
table(["버킷 (test)", "플래그율"] + [SHORT[l] for l in CONFIRMED],
      [[nm, rt(tt[key])[0]] + [T["flag_by_type"][b][l] for l in CONFIRMED] for b, nm, key in _bn], [3.0, 3.2] + [1.35] * 8, size=8)
table(["버킷 (dev)", "플래그율"] + [SHORT[l] for l in CONFIRMED],
      [[nm, rt(dt[key])[0]] + [D["flag_by_type"][b][l] for l in CONFIRMED] for b, nm, key in _bn], [3.0, 3.2] + [1.35] * 8, size=8)

h("읽을 때 알아 둘 것")
_gap = (D["macro_prauc"] - T["macro_prauc"]) if (D["macro_prauc"] == D["macro_prauc"] and T["macro_prauc"] == T["macro_prauc"]) else None
if _gap is not None:
    bullet(f"dev 와 test 의 macro PR-AUC 차이는 {_gap:+.3f} 다 (8종 기준 · dev {f3(D['macro_prauc'])} · test {f3(T['macro_prauc'])}).")
_seen = [v for v in T["by_prov"].values() if v["in_train"] and v["pos_recall"]["n"]]; _unseen = [v for v in T["by_prov"].values() if not v["in_train"] and v["pos_recall"]["n"]]
_agg = lambda vs: (sum(v["pos_recall"]["k"] for v in vs), sum(v["pos_recall"]["n"] for v in vs))
if _seen and _unseen:
    (k1, n1), (k2, n2) = _agg(_seen), _agg(_unseen)
    bullet(f"test 양성 {n1 + n2:,}행 중 {n2:,}행({n2 / (n1 + n2) * 100:.0f}%)은 학습에 없던 원천이다. 양성 탐지 재현율은 학습에 있던 원천 {k1 / n1 * 100:.1f}% · 없던 원천 {k2 / n2 * 100:.1f}% 다.")
_top = max(CONFIRMED, key=lambda l: T["flag_by_type"]["D"][l])
if tt["D_flag"]["k"]:
    bullet(f"D 플래그 {tt['D_flag']['k']:,}행에서 가장 많이 울린 유형은 {_top}({T['flag_by_type']['D'][_top]:,}건)이다. test 의 D 는 대부분 해설서 문구다.")
_low = [l for l in CONFIRMED if T["label"][l]["n"] >= 30 and T["label"][l]["R"] < 0.5]
if _low:
    bullet("재현율이 0.5 에 못 미친 유형(n ≥ 30): " + " · ".join(f"{l} {T['label'][l]['R']:.2f} (PR-AUC {f3(T['label'][l]['prauc'])})" for l in _low) + ".")
bullet("M 행은 학습에서 뺐다(D-296 개정 2). M 플래그율은 「보류해야 할 문구를 위반으로 울린 비율」로 읽는다 — 채점 지표가 아니고, v10(M 을 음성으로 학습)보다 높게 나올 수 있다.")

h("제한")
bullet("이 골든 판의 test 는 v10 에서 이미 채점했고 이번이 재채점이다. v10 → v11 의 변경(M 제외 · 8종)은 팀장 결정(D-296 개정 2 · D-321)을 옮긴 것이고 test 수치를 보고 고른 것이 아니다. 이 결과로 후보 · 설정 · threshold 를 다시 고르지 않는다(D-175). 백본 선택은 dev 수치로만 한다.")
bullet(f"평가 음성(조건 L)이 {tt['L_fpr']['n']}행이라 음성 오판정률은 " + ("측정 불가다(n<30 · D-301)." if tt["L_fpr"]["n"] < 30 else "참고 수준이다."))
bullet(f"유형 미정 C·A·B 행({DATA_STATS['removed_pending']}행 · 근거가 빈 대구청 판 5행 포함)은 학습에서 뺐다. 사례집 낱말 행은 v10 과 같이 학습에 두었다 — 낱말을 뺀 판과는 비교하지 않았다(전달 10-05 §5).")
bullet("출력 단위는 유형이다(app/encoder.py 가 읽는 형식). 호 단위로 학습해 유형으로 접는 판과는 비교하지 않았다(전달 10-05 §5 · 팀장 판정 대기).")
bullet("dev 는 학습과 같은 원천에서 뗀 묶음이다. 같은 원천 안의 다른 문서라 test 보다 쉬울 수 있다.")
bullet("이유 구역 판독(D-318)은 전부 모델 판독이다. 그 라벨로 학습 · dev 채점을 했다.")
bullet(f"새로 편입한 두 유형은 학습 양성의 주입(합성) 비중이 높다 — 부당_비교광고 {DATA_STATS['train_label_pos']['부당_비교광고']}행 중 {DATA_STATS['train_label_injected']['부당_비교광고']} · 비방광고 {DATA_STATS['train_label_pos']['비방광고']}행 중 {DATA_STATS['train_label_injected']['비방광고']}. threshold 는 합성이 없는 dev 로 골랐다.")
bullet(f"기능성화장품_오인은 편입 대기라 저장 모델에서 나오지 않는다(threshold {WAIT_THRESHOLD}). 학습 양성 {DATA_STATS['train_label_pos']['기능성화장품_오인']}행.")

h("다음 단계")
bullet("☐ kcbert · kcelectra 두 보고서의 dev 수치로 백본을 정한다 (dev macro PR-AUC 평균 ± 표준편차 · dev 양성 재현율 · dev D 플래그율)")
bullet("☐ 앱 모델 교체 여부 — 팀장 판정. 교체하면 3차 체크리스트 4단계(모델 교체) · 5단계(판정 로직 margin 재측정)를 따른다")
bullet("☐ 그래프 배선 — label_scheme.json 의 「train_m_excluded · train_cond_rows · golden_sha256」로 새 판 · M 제외 학습을 확인한다(D-296 개정 2)")
bullet("☐ 출력 단위(유형 대 호) · 사례집 낱말 행 포함 여부 — 팀장 판정 대기(전달 10-05 §5)")
bullet("☐ threshold 선택 규칙 사전 고정 여부 — 팀장 판정 (개선점 10-01 ④번)")

REPORT_NAME = f"판정인코더_파인튜닝_결과보고서_{NOTEBOOK_VERSION}_{BACKBONE}_{RUN_STAMP}.docx"
REPORT_PATH = os.path.join(OUT_DIR, REPORT_NAME)
doc.save(REPORT_PATH)
print("보고서:", REPORT_PATH)

## 10. 묶기 · 내려받기

In [ ]:
def zip_paths(zip_path, items):
    """items: (디스크 경로, 압축 안 이름). 모델 가중치는 이미 압축이 안 되므로 저장만 한다."""
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as z:
        for src, arc in items:
            z.write(src, arc, compress_type=zipfile.ZIP_STORED if src.endswith(".safetensors") else zipfile.ZIP_DEFLATED)
    return zip_path

BUNDLE = zip_paths(os.path.join(OUT_DIR, f"결과묶음_{RUN_ID}.zip"),
                   [(REPORT_PATH, REPORT_NAME), (os.path.join(OUT_DIR, "results.json"), "results.json"),
                    (os.path.join(SAVE_DIR, "label_scheme.json"), "label_scheme.json")] + [(p, os.path.basename(p)) for p in FIGS.values()])
MODEL_ZIP = zip_paths(os.path.join(OUT_DIR, f"{MODEL_DIRNAME}_{RUN_STAMP}.zip"),
                      [(os.path.join(SAVE_DIR, fn), f"{MODEL_DIRNAME}/{fn}") for fn in sorted(os.listdir(SAVE_DIR))])
DELIVER = [REPORT_PATH, BUNDLE, MODEL_ZIP]
for p in DELIVER:
    print(f"  {os.path.basename(p):70s} {os.path.getsize(p) / 1e6:8.1f} MB")

if DRIVE_RUNS and not SMOKE:
    try:
        dst = os.path.join(DRIVE_RUNS, RUN_ID); os.makedirs(dst, exist_ok=True)
        for p in DELIVER:
            shutil.copy(p, dst)
        print("Drive 보관:", dst)
    except Exception as e:
        print("Drive 보관 안 됨:", e)

t = TEST_RES["team"]
print(f"\n════════ 끝 · {RUN_ID} · {round((time.time() - T_START) / 60, 1)}분 ════════")
print(f"dev macro PR-AUC {np.mean(DEV_PRAUCS):.3f} ± {np.std(DEV_PRAUCS):.3f} · 선택 시드 {BEST_SEED}")
print(f"test 양성 재현율 {t['pos_recall']['rate']*100:.1f}% · D 플래그 {(t['D_flag']['rate'] or 0)*100:.1f}% · macro PR-AUC {TEST_RES['macro_prauc']:.3f}")
if IN_COLAB:
    for p in DELIVER:                     # 보고서 → 결과 묶음 → 모델(가장 크다) 순서
        colab_files.download(p); time.sleep(2)
    print("⬇ 내려받기를 시작했다 — 브라우저가 「여러 파일 다운로드」를 물으면 허용한다. 모델 zip 은 수 분 걸린다.")